# 02_dataset_maestro

Construcción del dataset maestro.

In [ ]:
from pathlib import Path
import unicodedata
import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_RAW = PROJECT_ROOT/'data'/'raw'
DATA_PROCESSED = PROJECT_ROOT/'data'/'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [3]:
pobreza = pd.read_excel(DATA_RAW/'pobreza_2024.xlsx')
micro = pd.read_excel(DATA_RAW/'acceso_microcredito_2024.xlsx')
prod = pd.read_excel(DATA_RAW/'acceso_productos_financieros_2024.xlsx')
atm = pd.read_excel(DATA_RAW/'atm_x_10000_adultos_2024.xlsx')
internet = pd.read_excel(DATA_RAW/'internet_hogares_2024.xlsx')

In [ ]:
def _quitar_tildes(texto: str) -> str:
    normalizado = unicodedata.normalize('NFD', str(texto))
    return ''.join(c for c in normalizado if unicodedata.category(c) != 'Mn')

_REEMPLAZOS_DEP = {
    'BOGOTA, D.C.': 'BOGOTA D.C.',
    'BOGOTA DC': 'BOGOTA D.C.',
    'SANTAFE DE BOGOTA D.C': 'BOGOTA D.C.',
    'SANTAFE DE BOGOTA D.C.': 'BOGOTA D.C.',
    'SAI': 'SAN ANDRES Y PROVIDENCIA',
    'SAN ANDRES': 'SAN ANDRES Y PROVIDENCIA',
    'SAN ANDRES Y PROVIDENCIA (SAI)': 'SAN ANDRES Y PROVIDENCIA',
    'ARCHIPIELAGO DE SAN ANDRES PROVIDENCIA Y SANTA CATALINA': 'SAN ANDRES Y PROVIDENCIA',
    'ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA': 'SAN ANDRES Y PROVIDENCIA',
}

def clean_dep(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip().str.upper().apply(_quitar_tildes)
    return s.replace(_REEMPLAZOS_DEP)

def set_dep(df: pd.DataFrame) -> pd.DataFrame:
    col = [c for c in df.columns if 'depart' in c.lower() or 'dpto' in c.lower()][0]
    df = df.copy()
    df['departamento'] = clean_dep(df[col])
    return df

pobreza  = set_dep(pobreza)
micro    = set_dep(micro)
prod     = set_dep(prod)
atm      = set_dep(atm)
internet = set_dep(internet)

In [5]:
pobreza=pobreza[['departamento','pobreza_2024']]
micro=micro[['departamento','acceso_microcredito_2024']]
prod=prod[['departamento','acceso_productos_financieros_2024']]
atm=atm[['departamento','atm_x_10000_adultos_2024']]
internet=internet[['departamento','internet_hogares_2024']]

In [ ]:
df = pobreza.merge(micro,    on='departamento', how='inner')\
            .merge(prod,     on='departamento', how='inner')\
            .merge(atm,      on='departamento', how='inner')\
            .merge(internet, on='departamento', how='inner')

print(f'Departamentos en el dataset final: {df["departamento"].nunique()}')
df.head()

In [7]:
print(df.isna().sum())
print(df['departamento'].nunique())

departamento                          0
pobreza_2024                         13
acceso_microcredito_2024              4
acceso_productos_financieros_2024     4
atm_x_10000_adultos_2024              4
internet_hogares_2024                 4
dtype: int64
37


In [8]:
df.to_csv(DATA_PROCESSED/'master_dataset.csv', index=False)